# Evaluación end-to-end de RAG

## Objetivos

Evaluarás conjuntamente la pregunta, los contextos recuperados desde PDFs reales, la respuesta, las citas y la referencia.

**Métricas que se calculan en este cuaderno:**

1. **Retrieval determinista:** `recall@3`, `precision@3` y MRR.
2. **Controles deterministas de respuesta:** cita válida y abstención correcta.
3. **Ragas con Gemini como juez:** `faithfulness`, `answer relevancy`, `context precision` y `context recall`.
4. **Operación:** tokens, costo, latencia, p50 y p95.

**Caso guiado:** `R02`. La evidencia de instalación está disponible, pero aparece después de un fragmento irrelevante. El objetivo es aprender a decidir si el problema está en retrieval, generación o eficiencia.

## 1. Métricas deterministas de retrieval: Recall@3, Precision@3 y MRR

Primero cargamos el eval set con sus páginas doradas, verificamos que los PDFs reales no cambiaron y ejecutamos el retriever. Esta sección **sí calcula métricas**, pero sólo para retrieval: `recall@3`, `precision@3` y MRR. Todavía no evalúa una respuesta generada por Gemini.

1. **Retrieval:** ¿entró evidencia relevante y en qué posición?
2. **Generación:** ¿la respuesta usa esa evidencia, responde la pregunta y evita inventar?
3. **Seguridad de la respuesta:** ¿incluye citas y se abstiene cuando no hay soporte?
4. **Operación:** ¿cuánto tarda y cuánto cuesta producir la respuesta?

Las métricas no compiten: cada una ayuda a localizar una causa distinta. La salida de esta sección debe leerse antes de juzgar al LLM.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
from metrics_course.datasets import load_rag_cases
from metrics_course.real_retrieval import verify_pdf_provenance
from metrics_course.retrieval import mean_reciprocal_rank, precision_at_k, recall_at_k, reciprocal_rank

cases = load_rag_cases()
provenance = pd.DataFrame(verify_pdf_provenance(cases))
display(provenance[['source', 'exists', 'hash_matches']])
guided_case_id = 'R02'
guided_case = next(case for case in cases if case['id'] == guided_case_id)

retrieval_rows = []
rankings = []
for case in cases:
    relevance = [
        context['relevant']
        for context in sorted(case['contexts'], key=lambda context: context['rank'])
    ]
    rankings.append(relevance)
    retrieval_rows.append({
        'case_id': case['id'],
        'category': case['category'],
        'question': case['question'],
        'recall@3': recall_at_k(relevance, 3),
        'precision@3': precision_at_k(relevance, 3),
        'reciprocal_rank': reciprocal_rank(relevance),
    })

retrieval_metrics = pd.DataFrame(retrieval_rows)
mrr = mean_reciprocal_rank([
    ranking for case, ranking in zip(cases, rankings) if case['category'] != 'trap'
])
print('Sección 1 — Métricas deterministas de retrieval sobre PDFs reales.')
display(retrieval_metrics.style.format({
    'recall@3': '{:.3f}', 'precision@3': '{:.3f}', 'reciprocal_rank': '{:.3f}'
}))
display(pd.DataFrame([{'metric': 'MRR', 'value': mrr}]).style.format({'value': '{:.3f}'}))
print('Recall@3: ¿llegó la evidencia? | Precision@3: ¿cuánto ruido entró? | MRR: ¿qué tan arriba apareció el primer resultado útil?')


### Interpretación de Recall@3, Precision@3 y MRR

Esta tabla evalúa únicamente retrieval. Para `R02`, `recall@3 = 1.000`: el top-3 contiene la página de instalación correcta. `precision@3 = 0.333`: sólo una de tres páginas es útil. *Reciprocal rank* = `0.500`: la primera página útil está en rango 2. **MRR** promedia ese valor entre las preguntas respondibles y resume el orden general del retriever.

**Qué hacer:** diagnosticar primero retrieval/ranking. La métrica de juez que aparezca después no debe borrar esta evidencia sobre la calidad del contexto. `R03` se trata por abstención y no debe interpretarse como un retrieval exitoso sólo porque `recall` use una convención para el caso sin relevantes.

## 2. Generar una respuesta RAG con Gemini

Esta sección **genera la respuesta que luego evaluaremos**. La siguiente celda arma un prompt con los contextos recuperados y llama Gemini. El modelo recibe la instrucción de responder sólo con esa evidencia, citar sus fuentes y abstenerse si no existe soporte. Se asume que `GOOGLE_API_KEY` y el modelo están configurados.

Si Gemini no puede ejecutarse por una configuración o dependencia, la celda muestra el error y usa explícitamente la respuesta versionada sólo para permitir inspeccionar el resto del flujo. No confunde ese respaldo con una generación nueva.

In [ ]:
from metrics_course.gemini import generate_rag_predictions
from metrics_course.operational import summarize_operations

try:
    generation_records = generate_rag_predictions(cases)
    generation_table = pd.DataFrame(generation_records)
    response_by_id = generation_table.set_index('case_id')['response'].to_dict()
    operation_summary = summarize_operations(generation_records)
    print('Generación real con Gemini — los registros siguientes corresponden a esta ejecución.')
except Exception as error:
    response_by_id = {case['id']: case['answer'] for case in cases}
    generation_table = pd.DataFrame({
        'case_id': [case['id'] for case in cases],
        'response': [response_by_id[case['id']] for case in cases],
    })
    operation_summary = None
    print('No se pudo generar con Gemini; se muestran respuestas versionadas como respaldo didáctico.')
    print(f'{type(error).__name__}: {error}')

print('Salida de generación — estas respuestas serán evaluadas en las secciones 3 y 5.')
display(generation_table[['case_id', 'response']])


### Cómo leer la respuesta generada

Esta salida todavía no asigna una puntuación; es el artefacto que evaluaremos en las secciones siguientes. En `R02`, una buena respuesta debe mencionar mangueras, suministro de agua y nivelación; esos son los hechos de la referencia y de la página 5 del manual de instalación. Si la respuesta usa sólo la primera página recuperada, el problema nace del ranking. Si recibe la página 5 y aun así inventa o ignora esos hechos, el problema es de generación.

**Qué hacer:** al ejecutar Gemini, lee primero `R02` antes de mirar promedios. Es el puente más claro entre la lista recuperada y la calidad de la respuesta.

## 3. Controles deterministas de respuesta: cita válida y abstención correcta

Aquí vuelve a empezar la evaluación. Antes de usar un juez LLM comprobamos dos contratos simples: para preguntas respondibles, la respuesta debe incluir una cita exacta de los contextos disponibles; para la pregunta trampa, debe declarar que no encontró evidencia suficiente. Estas comprobaciones no prueban fidelidad completa, pero detectan fallas visibles y auditables.

In [ ]:
def has_valid_context_citation(response, contexts):
    expected_citations = [
        f"[{context['source']}, p. {context['page']}]"
        for context in contexts
    ]
    return any(citation in response for citation in expected_citations)

quality_rows = []
for case in cases:
    response = response_by_id[case['id']]
    needs_abstention = case['category'] == 'trap'
    quality_rows.append({
        'case_id': case['id'],
        'citation_required': not needs_abstention,
        'valid_context_citation': has_valid_context_citation(response, case['contexts']),
        'abstention_required': needs_abstention,
        'correct_abstention': 'no encontré evidencia suficiente' in response.casefold(),
    })

response_quality = pd.DataFrame(quality_rows)
print('Evaluación determinista — citas válidas y abstención correcta.')
display(response_quality)

guided_quality = response_quality.loc[response_quality.case_id == guided_case_id].iloc[0]
print(f"{guided_case_id} tiene cita válida: {guided_quality['valid_context_citation']}")
trap_quality = response_quality.loc[response_quality.abstention_required].iloc[0]
print(f"{trap_quality.case_id} se abstuvo correctamente: {trap_quality['correct_abstention']}")
print('Una cita válida no prueba fidelidad completa; Ragas lo evaluará en la sección 5.')


### Interpretación de citas y abstención

Con las respuestas versionadas, `R02` tiene una cita válida y `R03` se abstiene correctamente. Esto es bueno: permite rastrear la evidencia y evita una respuesta fuera del corpus. Pero una cita válida sólo demuestra que aparece una referencia con formato correcto; no demuestra que todas las afirmaciones estén respaldadas. Esa es la razón de usar `faithfulness` después.

**Qué hacer:** si falla una cita o la abstención, corrige primero el prompt, la plantilla de respuesta o el umbral de recuperación. Si pasan ambos controles, continúa con la evaluación de juez.

## 4. Cómo se construye y valida un eval set humano

El juez LLM no decide qué evidencia es correcta. Antes de ejecutar Ragas, una persona debe construir el eval set: leer el PDF, escribir una pregunta, identificar la fuente y página que sí responden, redactar una respuesta de referencia y agregar preguntas trampa.

En este curso, cada caso guarda la fuente, página y hash SHA-256 del PDF dorado. El hash verifica que el archivo revisado por la persona no cambió. **Lo que el hash no verifica** es que la etiqueta sea correcta: esa decisión requiere revisión humana. En producción, además registra anotador, fecha, guía de etiquetado y una segunda revisión para desacuerdos.

In [ ]:
gold_annotations = []
for case in cases:
    for evidence in case['relevant_pages']:
        gold_annotations.append({
            'case_id': case['id'],
            'question': case['question'],
            'human_selected_source': evidence['source'],
            'human_selected_page': evidence['page'],
            'pdf_hash_recorded': evidence['pdf_sha256'][:12] + '…',
            'reference_answer': case['reference_answer'],
        })

gold_annotation_table = pd.DataFrame(gold_annotations)
print('Registro de etiquetas doradas revisadas por una persona: pregunta → PDF/página → respuesta de referencia.')
display(gold_annotation_table)
print('La tabla de procedencia de la sección 1 verifica que los hashes de estos PDFs coinciden.')


### Validación humana que debe ocurrir antes de confiar en métricas

1. Leer la página marcada y confirmar que realmente permite responder la pregunta.
2. Confirmar que la respuesta de referencia no agrega información ausente del PDF.
3. Marcar páginas cercanas pero no útiles como irrelevantes.
4. Incluir preguntas sin evidencia y definir la frase de abstención esperada.
5. Resolver desacuerdos entre revisores antes de calcular métricas.

Las métricas son confiables sólo en la medida en que estas etiquetas doradas sean confiables.

## 5. Prompt explícito de un LLM como juez: ejemplo de Faithfulness

Ragas construye sus propios prompts internos. Para que el contrato sea visible en clase, mostramos un prompt equivalente y simplificado para `faithfulness`. El juez recibe pregunta, respuesta, evidencia, reglas y un formato JSON obligatorio. No se le pide “¿te gusta esta respuesta?”; se le pide comprobar cada afirmación contra la evidencia.

In [ ]:
from IPython.display import Markdown, display
from metrics_course.rag_judge import build_faithfulness_judge_prompt

prompt_case = {
    **guided_case,
    'contexts': [
        {**context, 'text': context['text'][:700] + '…'}
        for context in guided_case['contexts']
    ],
}
judge_prompt_preview = build_faithfulness_judge_prompt(
    prompt_case, response_by_id[guided_case_id]
)
print('Prompt didáctico visible para evaluar Faithfulness en R02:')
display(Markdown(f'```text\n{judge_prompt_preview}\n```'))


### Cómo configurar un juez reproducible

Fija el modelo de juez, temperatura, versión del prompt y conjunto de evaluación. Si cambias cualquiera de ellos, el resultado puede cambiar y debe registrarse como una versión nueva de la evaluación. Nunca guardes la clave en el notebook ni en Git.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
judge_configuration = pd.DataFrame([
    {'setting': 'GOOGLE_API_KEY', 'value': 'configured' if os.getenv('GOOGLE_API_KEY') else 'missing', 'why_it_matters': 'Autoriza Gemini sin exponer la clave.'},
    {'setting': 'METRICS_JUDGE_MODEL', 'value': os.getenv('METRICS_JUDGE_MODEL', 'gemini-2.5-flash'), 'why_it_matters': 'El modelo forma parte de la versión del juez.'},
    {'setting': 'METRICS_EMBEDDING_MODEL', 'value': os.getenv('METRICS_EMBEDDING_MODEL', 'models/gemini-embedding-001'), 'why_it_matters': 'Answer relevancy usa embeddings.'},
    {'setting': 'judge_temperature', 'value': '0', 'why_it_matters': 'Reduce variación del juez.'},
])
print('Configuración del juez — la clave nunca se muestra.')
display(judge_configuration)


## 6. Preparar la entrada para las métricas Ragas con juez

Esta sección todavía **no ejecuta Ragas ni calcula sus puntuaciones**. Sólo prepara las cuatro piezas que Ragas necesita: `user_input`, `response`, `reference` y `retrieved_contexts`. En la siguiente sección, Ragas usa esas piezas para calcular: `faithfulness`, `answer relevancy`, `context precision` y `context recall`. Excluimos la pregunta trampa porque sus métricas de contexto no se diseñaron para un caso sin evidencia; la abstención ya se validó en la sección anterior.

In [ ]:
from metrics_course.rag_judge import build_ragas_rows, evaluate_with_ragas

judge_cases = [case for case in cases if case['category'] != 'trap']
ragas_input = pd.DataFrame(build_ragas_rows(judge_cases, response_by_id))
print('Entrada preparada para Ragas — todavía no hay puntuaciones de juez.')
display(ragas_input[['user_input', 'response', 'reference', 'retrieved_contexts']])


### Qué mide el juez y cómo leerlo

- **Faithfulness:** si las afirmaciones de la respuesta están respaldadas por los contextos. Bajo = posible alucinación o uso incorrecto del contexto.
- **Answer relevancy:** si la respuesta atiende la pregunta. Bajo = la respuesta se desvía, aunque sea fiel.
- **Context precision:** si los resultados útiles están arriba y el contexto evita ruido. Bajo = problema de ranking.
- **Context recall:** si los contextos contienen lo necesario para sostener la referencia. Bajo = problema de cobertura, corpus o recuperación.

Estas puntuaciones son estimaciones de un juez LLM, no una verdad absoluta. Registra modelo y fecha, conserva el eval set fijo e inspecciona los desacuerdos con la evidencia. Este curso no construye ni calibra un juez propio.

## 7. Métricas Ragas con Gemini como juez: Faithfulness, Answer Relevancy, Context Precision y Context Recall

Esta celda ejecuta directamente las puntuaciones de juez: `faithfulness`, `answer relevancy`, `context precision` y `context recall`. Asume que `GOOGLE_API_KEY` y las dependencias ya están configuradas. Ragas usa Gemini como juez con temperatura 0. La llamada tiene costo y se mide su latencia total. Si ocurre un problema de configuración o compatibilidad, se muestra un mensaje claro sin ocultar el error.

In [ ]:
import time

try:
    judge_started = time.perf_counter()
    ragas_result = evaluate_with_ragas(judge_cases, response_by_id)
    judge_latency_seconds = time.perf_counter() - judge_started
    judge_scores = pd.DataFrame(ragas_result)
    print('Evaluación con juez — Faithfulness, Answer Relevancy, Context Precision y Context Recall.')
    display(judge_scores)
    print(f'Latencia total del juez Ragas (s): {judge_latency_seconds:.3f}')
except Exception as error:
    judge_scores = None
    judge_latency_seconds = None
    print('No se pudo ejecutar Ragas con Gemini.')
    print(f'{type(error).__name__}: {error}')
    print('Revisa GOOGLE_API_KEY, el modelo configurado y la versión de Ragas; el resto del notebook puede seguir ejecutándose.')


### Cómo interpretar las puntuaciones de Ragas

Si la celda terminó correctamente, esta salida contiene métricas de juez. Lee cada fila antes del promedio. En `R02`, espera que `context precision` refleje el ruido en rango 1; si `context recall` es alto pero `faithfulness` bajo, la evidencia estaba disponible y el generador no la utilizó correctamente. Si ambos context metrics son bajos, mejora retrieval antes de modificar el prompt.

Una puntuación alta no elimina la revisión de hechos críticos. Usa los valores para priorizar inspecciones, no como un veredicto automático.

## 8. Métricas operativas: tokens, costo, latencia, p50 y p95

Esta sección responde: **¿cuánto costó y cuánto tardó generar las respuestas?** Sólo muestra datos cuando se ejecutó Gemini en vivo; no crea números ficticios en modo sin API.

In [ ]:
if operation_summary is not None:
    print('Evaluación operativa — costo y latencia de respuestas generadas en vivo.')
    display(generation_table[['case_id', 'input_tokens', 'output_tokens', 'cost_usd', 'latency_seconds']])
    display(pd.DataFrame([operation_summary]))
    print(
        f"Interpretación: p50={operation_summary['latency_p50_seconds']:.3f}s, "
        f"p95={operation_summary['latency_p95_seconds']:.3f}s y "
        f"costo total estimado=USD {operation_summary['cost_usd']:.8f}."
    )
else:
    print('Sin métricas operativas: ejecuta la generación en vivo para registrar costo y latencia reales.')


### Cómo interpretar costo y latencia

- **Costo total:** gasto estimado de las respuestas de esta ejecución.
- **p50:** tiempo típico; aproximadamente la mitad de las llamadas tarda menos.
- **p95:** experiencia lenta; aproximadamente 95 de cada 100 llamadas deberían tardar menos.

Un p95 bajo no vuelve correcta una respuesta, y una respuesta excelente puede no ser viable si su costo o latencia superan el presupuesto del producto.

## 9. Diagnóstico: decidir qué componente ajustar

Esta sección combina las señales disponibles. Si activaste Ragas, une retrieval, juez y latencia por caso. Si no activaste Ragas, muestra una guía de diagnóstico sin inventar puntuaciones de juez.

In [ ]:
from metrics_course.reporting import diagnose_rag

if judge_scores is not None:
    print('Diagnóstico combinado — retrieval determinista + métricas de juez.')
    judge_scores = judge_scores.copy()
    judge_scores['case_id'] = [case['id'] for case in judge_cases]
    retrieval_for_diagnosis = retrieval_metrics.rename(columns={
        'recall@3': 'recall_at_k', 'precision@3': 'precision_at_k'
    })
    diagnostic_input = retrieval_for_diagnosis.merge(judge_scores, on='case_id', how='inner')
    if operation_summary is not None:
        diagnostic_input = diagnostic_input.merge(
            generation_table[['case_id', 'latency_seconds']], on='case_id', how='left'
        )
    else:
        diagnostic_input['latency_seconds'] = 0.0
    display(diagnose_rag(diagnostic_input))
else:
    diagnostic_guide = pd.DataFrame([
        {'signal': 'Context recall bajo', 'first_area_to_check': 'Corpus, chunking, embeddings o consulta.'},
        {'signal': 'Context precision bajo o MRR bajo', 'first_area_to_check': 'Ranking, filtros o re-ranking.'},
        {'signal': 'Faithfulness bajo con contexto suficiente', 'first_area_to_check': 'Prompt, modelo o generación.'},
        {'signal': 'p95/costo alto', 'first_area_to_check': 'Modelo, top-k, tamaño de contexto o concurrencia.'},
    ])
    display(diagnostic_guide)


### Interpretación del diagnóstico: de métricas a una decisión

Para `R02`, la evidencia determinista ya apunta a ranking: recall completo con ruido y primer relevante en rango 2. Si Ragas confirma `context precision` bajo, el siguiente experimento es re-rankear. Sólo si `faithfulness` o `answer relevancy` fallan pese a tener buen contexto conviene intervenir la generación.

**Qué hacer:** conserva por caso la pregunta, contextos, respuesta, puntuaciones y versión del modelo. Así una mejora de calidad que aumente p95 o costo también puede discutirse como una decisión de producto.

## Resumen final y métricas mínimas para producción

Un RAG mínimo debe reportar, por versión del sistema:

- Retrieval: `recall@k`, `precision@k` y MRR sobre preguntas con evidencia.
- Generación: `faithfulness` y `answer relevancy`, revisadas por caso además de promedios.
- Seguridad: porcentaje de citas válidas y éxito de abstención en preguntas trampa.
- Operación: costo por respuesta, p50 y p95 de latencia.

1. Ejecuta Gemini en vivo, conserva la tabla de `R02` y explica si el fallo observado pertenece a retrieval o generación.
2. Mueve `C5` de `R02` a rango 1. Predice qué ocurrirá con MRR, context precision y la respuesta.
3. Añade una respuesta que tenga una cita válida pero afirme un dato ausente. ¿Qué control la detecta y cuál no?

**Cierre:** la evaluación es parte del producto. Un número aislado no da un veredicto; casos versionados, métricas complementarias e inspección de fallas permiten mejorar con evidencia.